# Validação temporal com janela expansiva

Após os ajustes de hiperparâmetros, será realizada uma validação temporal com janela expansiva, tendo como objetivo verificar se o modelo mantém desempenho estável ao longo do tempo.

A cada rodada, o modelo será treinado com dados de anos anteriores e avaliado no ano seguinte. Dessa forma, será possível observar se os padrões aprendidos pelo modelo continuam úteis em períodos futuros.

As janelas avaliadas serão:

* Treino em 2020 e validação em 2021;

* Treino em 2020–2021 e validação em 2022;

* Treino em 2020–2022 e validação em 2023;

* Treino em 2020–2023 e validação em 2024.

Em cada janela, o modelo será treinado novamente do zero com os mesmos hiperparâmetros selecionados na fase anterior. Antes do treinamento, será aplicada a padronização e o balanceamento nas bases utilizadas para treino. Já a base de validação ou teste será apenas padronizada com o mesmo scaler ajustado no treino, sem aplicação de balanceamento, representando melhor a distribuição real dos dados.


In [ ]:
# Manipulação de caminhos e arquivos
from pathlib import Path

# Manipulação de dados
import pandas as pd

# Balanceamento da classe minoritária
from imblearn.over_sampling import SMOTE

# Modelo de classificação utilizado
from sklearn.linear_model import LogisticRegression

# Padronização das variáveis numéricas para a Regressão Logística
from sklearn.preprocessing import StandardScaler

# Métricas de avaliação do modelo
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score
)


Vou carregar as bases que serão utilizadas na validação temporal com janela expansiva.


In [ ]:
# Caminho da pasta com as bases separadas para treino e teste
pasta_saida = Path("dados/processed/splits")

# Como será usada Regressão Logística, o SMOTE será aplicado após a padronização.
# Por isso, não será carregada a base já balanceada.


In [ ]:
# Janela temporal 1: treino em 2020 e teste em 2021

# Bases de treino com dados de 2020
x_treino_2020 = pd.read_csv(pasta_saida / "x_treino_2020.csv")
y_treino_2020 = pd.read_csv(
    pasta_saida / "y_treino_2020.csv"
)["internacao_prolongada"]

# Bases de teste com dados de 2021
x_teste_2021 = pd.read_csv(pasta_saida / "x_teste_2021.csv")
y_teste_2021 = pd.read_csv(
    pasta_saida / "y_teste_2021.csv"
)["internacao_prolongada"]


In [ ]:
# Janela temporal 2: treino em 2020 e 2021, teste em 2022

# Bases de treino com dados de 2020 e 2021
x_treino_2020_2021 = pd.read_csv(pasta_saida / "x_treino_2020_2021.csv")
y_treino_2020_2021 = pd.read_csv(
    pasta_saida / "y_treino_2020_2021.csv"
)["internacao_prolongada"]

# Bases de teste com dados de 2022
x_teste_2022 = pd.read_csv(pasta_saida / "x_teste_2022.csv")
y_teste_2022 = pd.read_csv(
    pasta_saida / "y_teste_2022.csv"
)["internacao_prolongada"]


In [ ]:
# Janela temporal 3: treino de 2020 a 2022, teste em 2023

# Bases de treino com dados de 2020, 2021 e 2022
x_treino_2020_2022 = pd.read_csv(pasta_saida / "x_treino_2020_2022.csv")
y_treino_2020_2022 = pd.read_csv(
    pasta_saida / "y_treino_2020_2022.csv"
)["internacao_prolongada"]

# Bases de teste com dados de 2023
x_teste_2023 = pd.read_csv(pasta_saida / "x_teste_2023.csv")
y_teste_2023 = pd.read_csv(
    pasta_saida / "y_teste_2023.csv"
)["internacao_prolongada"]


In [ ]:
# Janela temporal final: treino de 2020 a 2023, teste em 2024

# Bases de treino com dados de 2020, 2021, 2022 e 2023
x_treino_2020_2023 = pd.read_csv(pasta_saida / "x_treino_2020_2023.csv")
y_treino_2020_2023 = pd.read_csv(
    pasta_saida / "y_treino_2020_2023.csv"
)["internacao_prolongada"]

# Bases de teste com dados de 2024
x_teste_2024 = pd.read_csv(pasta_saida / "x_teste_2024.csv")
y_teste_2024 = pd.read_csv(
    pasta_saida / "y_teste_2024.csv"
)["internacao_prolongada"]


Vou padronizar as bases, ajustando o scaler apenas nos dados de treino de cada janela temporal e aplicando a mesma transformação na respectiva base de validação. Dessa forma, evito vazamento de informação e mantenho a avaliação mais próxima de um cenário real.


In [ ]:
# Variáveis numéricas que serão padronizadas em cada janela temporal
colunas_escalar = [
    "idade_anos",
    "ano_internacao",
    "mes_internacao"
]


In [ ]:
# Janela 1: treino em 2020 e teste em 2021

# Instancia o scaler específico da primeira janela temporal
scaler_2020_2021 = StandardScaler()

# Cria cópias das bases para preservar os dados originais
x_treino_2020_padronizado = x_treino_2020.copy()
x_teste_2021_padronizado = x_teste_2021.copy()

# Ajusta o scaler apenas na base de treino de 2020
# e aplica a padronização nos dados de treino.
x_treino_2020_padronizado[colunas_escalar] = scaler_2020_2021.fit_transform(
    x_treino_2020[colunas_escalar]
)

# Aplica no teste de 2021 a mesma transformação aprendida no treino.
# Isso evita vazamento de informação da base de teste.
x_teste_2021_padronizado[colunas_escalar] = scaler_2020_2021.transform(
    x_teste_2021[colunas_escalar]
)


In [ ]:
# Janela 2: treino em 2020-2021 e teste em 2022

# Instancia o scaler específico da segunda janela temporal
scaler_2020_2021_2022 = StandardScaler()

# Cria cópias das bases para preservar os dados originais
x_treino_2020_2021_padronizado = x_treino_2020_2021.copy()
x_teste_2022_padronizado = x_teste_2022.copy()

# Ajusta o scaler apenas na base de treino acumulada de 2020-2021
# e aplica a padronização nos dados de treino.
x_treino_2020_2021_padronizado[colunas_escalar] = scaler_2020_2021_2022.fit_transform(
    x_treino_2020_2021[colunas_escalar]
)

# Aplica no teste de 2022 a mesma transformação aprendida no treino.
# Isso evita vazamento de informação da base de teste.
x_teste_2022_padronizado[colunas_escalar] = scaler_2020_2021_2022.transform(
    x_teste_2022[colunas_escalar]
)


In [ ]:
# Janela 3: treino em 2020-2022 e teste em 2023

# Instancia o scaler ajustado apenas com os dados de treino
scaler_2020_2022 = StandardScaler()

# Cria cópias das bases para preservar os dados originais
x_treino_2020_2022_padronizado = x_treino_2020_2022.copy()
x_teste_2023_padronizado = x_teste_2023.copy()

# Ajusta o scaler na base de treino de 2020-2022
x_treino_2020_2022_padronizado[colunas_escalar] = scaler_2020_2022.fit_transform(
    x_treino_2020_2022[colunas_escalar]
)

# Aplica no teste de 2023 a transformação aprendida no treino
x_teste_2023_padronizado[colunas_escalar] = scaler_2020_2022.transform(
    x_teste_2023[colunas_escalar]
)


In [ ]:
# Janela final: treino em 2020-2023 e teste em 2024

# Instancia o scaler ajustado apenas com os dados de treino final
scaler_2020_2023 = StandardScaler()

# Cria cópias das bases para preservar os dados originais
x_treino_2020_2023_padronizado = x_treino_2020_2023.copy()
x_teste_2024_padronizado = x_teste_2024.copy()

# Ajusta o scaler na base de treino de 2020-2023
x_treino_2020_2023_padronizado[colunas_escalar] = scaler_2020_2023.fit_transform(
    x_treino_2020_2023[colunas_escalar]
)

# Aplica no teste de 2024 a transformação aprendida no treino
x_teste_2024_padronizado[colunas_escalar] = scaler_2020_2023.transform(
    x_teste_2024[colunas_escalar]
)


Vou balancear apenas as bases de treino de cada janela temporal utilizando SMOTE. As bases de validação não serão balanceadas, para manter a distribuição real dos dados e permitir uma avaliação mais fiel do desempenho do modelo.


In [ ]:
# Instancia o SMOTE para balancear a classe minoritária
smote = SMOTE(random_state=42)


In [ ]:
# Janela 1: aplica SMOTE apenas no treino de 2020 após a padronização.
# A base de teste de 2021 permanece sem balanceamento para simular dados reais.
x_treino_2020_padronizado_smote, y_treino_2020_padronizado_smote = smote.fit_resample(
    x_treino_2020_padronizado,
    y_treino_2020
)


In [ ]:
# Janela 2: aplica SMOTE apenas no treino acumulado de 2020-2021 após a padronização.
# A base de teste de 2022 permanece sem balanceamento para simular dados reais.
x_treino_2020_2021_padronizado_smote, y_treino_2020_2021_padronizado_smote = smote.fit_resample(
    x_treino_2020_2021_padronizado,
    y_treino_2020_2021
)


In [ ]:
# Janela 3: aplica SMOTE apenas no treino acumulado de 2020-2022 após a padronização.
# A base de teste de 2023 permanece sem balanceamento para simular dados reais.
x_treino_2020_2022_padronizado_smote, y_treino_2020_2022_padronizado_smote = smote.fit_resample(
    x_treino_2020_2022_padronizado,
    y_treino_2020_2022
)


In [ ]:
# Janela final: aplica SMOTE apenas no treino acumulado de 2020-2023 após a padronização.
# A base de teste de 2024 permanece sem balanceamento para simular dados reais.
x_treino_2020_2023_padronizado_smote, y_treino_2020_2023_padronizado_smote = smote.fit_resample(
    x_treino_2020_2023_padronizado,
    y_treino_2020_2023
)


# Janela 1: treino 2020 → teste 2021

In [ ]:
# Instancia a Regressão Logística com os hiperparâmetros finais escolhidos.
# O solver="saga" é adequado para bases maiores e suporta regularização L2.
# O penalty="l2" aplica regularização para reduzir overfitting.
# O C=5.0 controla a força da regularização.
# O max_iter=3000 aumenta o limite de iterações para ajudar na convergência.
# O tol=0.0001 define a tolerância usada como critério de parada.
# O random_state=42 garante reprodutibilidade dos resultados.
modelo_temporal_2021 = LogisticRegression(
    solver="saga",
    penalty="l2",
    C=5.0,
    max_iter=3000,
    tol=0.0001,
    random_state=42
)

# Treina o modelo com a base de 2020 padronizada e balanceada com SMOTE.
modelo_temporal_2021.fit(
    x_treino_2020_padronizado_smote,
    y_treino_2020_padronizado_smote
)


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",5.0
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'saga'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",3000
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net m

Testo o modelo.

In [ ]:
# Gera as previsões para a base de teste
y_pred_2021 = modelo_temporal_2021.predict(x_teste_2021_padronizado)


Avalio o modelo.


In [ ]:
# Avalia o desempenho do modelo na base de teste
print('Matriz de confusão:')
print(confusion_matrix(y_teste_2021, y_pred_2021))

print('\nClassification Report:')
print(classification_report(y_teste_2021, y_pred_2021))

print('Accuracy:', accuracy_score(y_teste_2021, y_pred_2021))
print('Recall:', recall_score(y_teste_2021, y_pred_2021))
print('Precision:', precision_score(y_teste_2021, y_pred_2021))
print('F1-score:', f1_score(y_teste_2021, y_pred_2021))


Matriz de confusão:
[[117268  31830]
 [ 13186  32021]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.79      0.84    149098
           1       0.50      0.71      0.59     45207

    accuracy                           0.77    194305
   macro avg       0.70      0.75      0.71    194305
weighted avg       0.81      0.77      0.78    194305

Accuracy: 0.7683229973495278
Recall: 0.7083195080407901
Precision: 0.501495669605801
F1-score: 0.5872288140255644


É possível observar que o modelo identificou corretamente 32.021 internações prolongadas em 2021, enquanto deixou passar 13.186 casos reais da classe 1, tendo um recall de aproximadamente 70,83%, uma precision de aproximadamente 50,15% e um F1-score de aproximadamente 58,72%.

De forma geral, o resultado foi positivo. O modelo apresentou boa capacidade de identificar internações prolongadas, mantendo uma precision moderada. Apesar da presença de falsos positivos, o desempenho sugere que o modelo conseguiu generalizar de 2020 para 2021 de forma satisfatória.


# Janela 2: treino 2020–2021 → teste 2022

In [ ]:
# Instancia a Regressão Logística com os hiperparâmetros finais escolhidos.
# O solver="saga" é adequado para bases maiores e suporta regularização L2.
# O penalty="l2" aplica regularização para reduzir overfitting.
# O C=5.0 controla a força da regularização.
# O max_iter=3000 aumenta o limite de iterações para ajudar na convergência.
# O tol=0.0001 define a tolerância usada como critério de parada.
# O random_state=42 garante reprodutibilidade dos resultados.
modelo_temporal_2022 = LogisticRegression(
    solver="saga",
    penalty="l2",
    C=5.0,
    max_iter=3000,
    tol=0.0001,
    random_state=42
)

# Treina o modelo com a base acumulada de 2020-2021,
# já padronizada e balanceada com SMOTE.
modelo_temporal_2022.fit(
    x_treino_2020_2021_padronizado_smote,
    y_treino_2020_2021_padronizado_smote
)


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",5.0
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'saga'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",3000
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net m

Testo o modelo

In [ ]:
# Gera as previsões para a base de teste
y_pred_2022 = modelo_temporal_2022.predict(x_teste_2022_padronizado)


Avalio o modelo

In [ ]:
# Avalia o desempenho do modelo na base de teste
print('Matriz de confusão:')
print(confusion_matrix(y_teste_2022, y_pred_2022))

print('\nClassification Report:')
print(classification_report(y_teste_2022, y_pred_2022))

print('Accuracy:', accuracy_score(y_teste_2022, y_pred_2022))
print('Recall:', recall_score(y_teste_2022, y_pred_2022))
print('Precision:', precision_score(y_teste_2022, y_pred_2022))
print('F1-score:', f1_score(y_teste_2022, y_pred_2022))


Matriz de confusão:
[[131290  33023]
 [ 12921  30368]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.80      0.85    164313
           1       0.48      0.70      0.57     43289

    accuracy                           0.78    207602
   macro avg       0.69      0.75      0.71    207602
weighted avg       0.82      0.78      0.79    207602

Accuracy: 0.7786919201163766
Recall: 0.7015177065767285
Precision: 0.4790585414333265
F1-score: 0.569328833895763


O modelo identificou corretamente 30.368 internações prolongadas em 2022, deixando escapar 12.921 casos reais da classe 1, tendo um recall de aproximadamente 70,15%, precision de aproximadamente 47,91% e F1-score de 56,93%.

De forma geral, o desempenho da segunda janela ainda pode ser considerado estável, pois o recall permaneceu próximo de 70%. Apesar disso, a queda no F1-score indica que será importante observar as próximas janelas para verificar se essa redução representa apenas uma variação anual ou uma tendência de perda de desempenho ao longo do tempo.


# Janela 3: treino 2020–2022 → teste 2023

In [ ]:
# Instancia a Regressão Logística com os hiperparâmetros finais escolhidos.
# O solver="saga" é adequado para bases maiores e suporta regularização L2.
# O penalty="l2" aplica regularização para reduzir overfitting.
# O C=5.0 controla a força da regularização.
# O max_iter=3000 aumenta o limite de iterações para ajudar na convergência.
# O tol=0.0001 define a tolerância usada como critério de parada.
# O random_state=42 garante reprodutibilidade dos resultados.
modelo_temporal_2023 = LogisticRegression(
    solver="saga",
    penalty="l2",
    C=5.0,
    max_iter=3000,
    tol=0.0001,
    random_state=42
)

# Treina o modelo com a base acumulada de 2020-2022,
# já padronizada e balanceada com SMOTE.
modelo_temporal_2023.fit(
    x_treino_2020_2022_padronizado_smote,
    y_treino_2020_2022_padronizado_smote
)


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",5.0
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'saga'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",3000
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net m

Testo o modelo

In [ ]:
# Gera as previsões para a base de teste
y_pred_2023 = modelo_temporal_2023.predict(x_teste_2023_padronizado)


Avalio o modelo

In [ ]:
# Avalia o desempenho do modelo na base de teste
print('Matriz de confusão:')
print(confusion_matrix(y_teste_2023, y_pred_2023))

print('\nClassification Report:')
print(classification_report(y_teste_2023, y_pred_2023))

print('Accuracy:', accuracy_score(y_teste_2023, y_pred_2023))
print('Recall:', recall_score(y_teste_2023, y_pred_2023))
print('Precision:', precision_score(y_teste_2023, y_pred_2023))
print('F1-score:', f1_score(y_teste_2023, y_pred_2023))


Matriz de confusão:
[[145986  36171]
 [ 14802  32386]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.80      0.85    182157
           1       0.47      0.69      0.56     47188

    accuracy                           0.78    229345
   macro avg       0.69      0.74      0.71    229345
weighted avg       0.82      0.78      0.79    229345

Accuracy: 0.7777453181887549
Recall: 0.6863185555649741
Precision: 0.47239523316364485
F1-score: 0.5596094863709016


O modelo identificou corretamente 32.386 internações prolongadas em 2023, enquanto deixou de identificar 14.802 casos reais da classe 1, tendo um recall de aproximadamente 68,63%, precision de aproximadamente 47,24% e F1-score de 55,96%.

De forma geral, a terceira janela ainda apresenta desempenho aceitável, mas reforça a importância de avaliar o modelo no final de 2024 para verificar se essa queda representa uma variação anual ou uma tendência de perda gradual de desempenho.


# Janela final: treino 2020–2023 → teste 2024

In [ ]:
# Instancia a Regressão Logística com os hiperparâmetros finais escolhidos.
# O solver="saga" é adequado para bases maiores e suporta regularização L2.
# O penalty="l2" aplica regularização para reduzir overfitting.
# O C=5.0 controla a força da regularização.
# O max_iter=3000 aumenta o limite de iterações para ajudar na convergência.
# O tol=0.0001 define a tolerância usada como critério de parada.
# O random_state=42 garante reprodutibilidade dos resultados.
modelo_temporal_2024 = LogisticRegression(
    solver="saga",
    penalty="l2",
    C=5.0,
    max_iter=3000,
    tol=0.0001,
    random_state=42
)

# Treina o modelo com a base acumulada de 2020-2023,
# já padronizada e balanceada com SMOTE.
modelo_temporal_2024.fit(
    x_treino_2020_2023_padronizado_smote,
    y_treino_2020_2023_padronizado_smote
)


c:\Users\djmet\OneDrive\Projetos\Ebac\Semantix\Projeto\SUS\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",5.0
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'saga'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",3000
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net m

Testo o modelo

In [ ]:
# Gera as previsões para a base de teste
y_pred_2024 = modelo_temporal_2024.predict(x_teste_2024_padronizado)


Avalio o modelo

In [ ]:
# Avalia o desempenho do modelo na base de teste
print('Matriz de confusão:')
print(confusion_matrix(y_teste_2024, y_pred_2024))

print('\nClassification Report:')
print(classification_report(y_teste_2024, y_pred_2024))

print('Accuracy:', accuracy_score(y_teste_2024, y_pred_2024))
print('Recall:', recall_score(y_teste_2024, y_pred_2024))
print('Precision:', precision_score(y_teste_2024, y_pred_2024))
print('F1-score:', f1_score(y_teste_2024, y_pred_2024))


Matriz de confusão:
[[146681  39136]
 [ 13821  33365]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.79      0.85    185817
           1       0.46      0.71      0.56     47186

    accuracy                           0.77    233003
   macro avg       0.69      0.75      0.70    233003
weighted avg       0.82      0.77      0.79    233003

Accuracy: 0.7727196645536752
Recall: 0.7070953248844997
Precision: 0.46020054895794543
F1-score: 0.5575375771804791


O modelo identificou corretamente 33.365 internações prolongadas em 2024, enquanto deixou de identificar 13.821 casos reais, tendo um recall de aproximadamente 70,71%, precision de aproximadamente 46,02% e F1-score de aproximadamente 55,75%.

De forma geral, a validação temporal mostra que o modelo é estável o suficiente para identificar internações prolongadas em anos futuros, mas ainda possui limitação na precisão das previsões positivas. Assim, o modelo pode ser útil como ferramenta de triagem inicial de risco, desde que seus alertas sejam interpretados como apoio à decisão e não como classificação definitiva.


# Conclusão Final

Após a comparação inicial entre diferentes algoritmos, ajustes básicos de hiperparâmetros, refinamento do modelo vencedor e validação temporal com janela expansiva, o modelo final foi um modelo de **Regressão Logística**, em que a base usada para treino passou por **padronização** e **balanceamento**. A configuração utilizada foi:

• **solver="saga"**;

• **penalty="l2"**;

• **C=5.0**;

• **max_iter=3000**;

• **tol=0.0001**.

Esse modelo foi escolhido por apresentar o melhor equilíbrio geral entre *recall*, *precision* e *F1-score* entre as versões refinadas da Regressão Logística. Além disso, na validação temporal, manteve desempenho relativamente estável ao longo dos anos avaliados.

A validação temporal mostrou que a acurácia permaneceu próxima de 77% nas diferentes janelas, enquanto o recall da classe 1 ficou próximo de 70%, indicando que o modelo conseguiu manter uma boa capacidade de identificar internações prolongadas mesmo quando aplicado a anos futuros. No entanto, a precision apresentou queda gradual ao longo das janelas, indicando que o modelo ainda gera uma quantidade relevante de falsos positivos. Em outras palavras, o modelo consegue encontrar boa parte das internações prolongadas, mas ainda classifica parte das internações não prolongadas como prolongadas.

De forma geral, o modelo pode ser considerado útil como ferramenta de triagem inicial de risco, não devendo ser interpretado como uma decisão automática definitiva, mas sim como um apoio à tomada de decisão, ajudando a sinalizar internações com maior chance de permanência prolongada.

O modelo tem entre suas principais limitações o uso de dados administrativos, a ausência de variáveis clínicas mais detalhadas, a possibilidade de mudanças no padrão de registro ao longo dos anos e a presença de falsos positivos. Como próximos passos, seria possível testar novos recortes temporais, incluir novas variáveis, avaliar outros thresholds de decisão e comparar o desempenho em diferentes estados ou regiões.

Assim sendo, o projeto demonstrou que é possível utilizar dados públicos do SIH/SUS para construir um modelo preditivo capaz de identificar internações com maior risco de permanência prolongada, com desempenho estável ao longo do tempo e potencial aplicação como ferramenta de apoio à gestão hospitalar.
